In [1]:
import os

# Ensure all relative paths (gw/, lookup/, reports/) resolve from the project root,
# regardless of whether the notebook is opened from notebooks/ or notebooks/transcripts_processor/.
cwd = os.getcwd()
if os.path.basename(cwd) == "text_processor":
    os.chdir("../..")

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/augustustranter/Documents/Programming/Projects/fpl/fpl_agent


In [2]:
# This variable indicates the gameweek for which we are generating a report.
# E.g. if gameweek = 3, transcripts in gw/gw3/transcripts/ are analysed.
gameweek = 3

In [3]:
import importlib.util
from pathlib import Path

filter_text_path = Path("notebooks/text_processor/1_filter_text.py")
filter_spec = importlib.util.spec_from_file_location("text_filter", filter_text_path)
assert filter_spec is not None and filter_spec.loader is not None
filter_module = importlib.util.module_from_spec(filter_spec)
filter_spec.loader.exec_module(filter_module)
process_text_folder = filter_module.process_text_folder

text_cache_dir = Path(f"gw/gw{gameweek}/text_cache")
filter_cache_dir = text_cache_dir / "filter_cache"
segment_words = 10
max_workers = 4

filtered_text_sources = process_text_folder(
    gameweek=gameweek,
    cache_dir=filter_cache_dir,
    segment_words=segment_words,
    max_workers=max_workers,
    refresh=False,
 )

filtered_text_payload = {
    source_name: filtered.text
    for source_name, filtered in filtered_text_sources.items()
}

print(f"Filter cache written to: {filter_cache_dir}")
print(f"Loaded {len(filtered_text_sources)} filtered text source(s) into memory")
print(f"Segment target word count: {segment_words}")
print(f"Parallel workers: {max_workers}")
print(sorted(filtered_text_payload))

Filter cache written to: gw/gw3/text_cache/filter_cache
Loaded 16 filtered text source(s) into memory
Segment target word count: 10
Parallel workers: 4
['batch_01', 'batch_02', 'batch_03', 'batch_04', 'batch_05', 'batch_06', 'batch_07', 'batch_08', 'batch_09', 'batch_10', 'batch_11', 'batch_12', 'batch_13', 'batch_14', 'batch_15', 'batch_16']


In [ ]:
import importlib.util
import json
from pathlib import Path

extract_text_path = Path("notebooks/text_processor/2_extract_text.py")
extract_spec = importlib.util.spec_from_file_location("text_extract", extract_text_path)
assert extract_spec is not None and extract_spec.loader is not None
extract_module = importlib.util.module_from_spec(extract_spec)
extract_spec.loader.exec_module(extract_module)
build_combined_output = extract_module.build_combined_output
process_filtered_text_sources = extract_module.process_filtered_text_sources

text_extraction_cache_dir = text_cache_dir / "extraction_cache"
text_extraction_output_path = Path(f"gw/gw{gameweek}/data/gw_{gameweek}_text_extractions.json")
extraction_model = "gpt-4.1-mini"
extraction_max_workers = 4
extraction_batch_size = 4

text_extraction_results = process_filtered_text_sources(
    filtered_text_payload,
    cache_dir=text_extraction_cache_dir,
    model=extraction_model,
    max_workers=extraction_max_workers,
    batch_size=extraction_batch_size,
    refresh=False,
)

combined_text_extraction = build_combined_output(gameweek, text_extraction_results)
text_extraction_output_path.parent.mkdir(parents=True, exist_ok=True)
text_extraction_output_path.write_text(
    json.dumps(combined_text_extraction, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Extraction cache written to: {text_extraction_cache_dir}")
print(f"Combined extraction JSON written to: {text_extraction_output_path}")
print(f"Extracted {len(combined_text_extraction['recommendations'])} recommendation(s)")
print(f"Extracted {len(combined_text_extraction['captaincy'])} captaincy mention(s)")
print(f"Processed {len(text_extraction_results)} source(s) in batches of {extraction_batch_size}")

Extraction cache written to: gw/gw3/text_cache/extraction_cache
Combined extraction JSON written to: gw/gw3/data/gw_3_text_extractions.json
Extracted 55 recommendation(s)
Extracted 16 captaincy mention(s)
Processed 16 source(s) in batches of 4
